# Codify 101: a statute as data

What Codify makes of one act: a jurisdiction's configuration, a structured text parsed
into Akoma Ntoso, the typed document behind that XML, schema validation, and the
model-driven structuring pass that turns plain text into all of the above.

Every section but the last runs offline. The last needs a chat model: copy
`.env.example` to `.env` at the repo root and add a key. Setup:

```
uv sync --group dev --extra migrations --extra serve --extra mcp
```

Then open this file in an editor that runs notebooks (VS Code does), with the repo's
`.venv` as the kernel.

The material is synthetic: `xa` is Atlantis, a fictional common-law jurisdiction that
ships with the repo so nothing here quotes a real statute book.

In [1]:
import logging
from pathlib import Path

import langfuse  # noqa: F401, its import resets the logger quieted below
import structlog
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # the repo's .env; exported variables win
logging.getLogger("langfuse").setLevel(logging.ERROR)  # tracing is optional
structlog.configure(  # warnings only, uncoloured
    wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING),
    processors=[structlog.processors.add_log_level, structlog.dev.ConsoleRenderer(colors=False)],
)
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
FIXTURES = REPO / "tests" / "fixtures" / "synthetic"

## 1. A jurisdiction is configuration

`data/jurisdictions/<code>/config.json` tells the pipeline what a document from that
jurisdiction looks like: which document types exist, how each is nested, what an
enacting formula reads like. Nothing about a jurisdiction is hard-coded.

In [2]:
from codify.jurisdictions import load_config

cfg = load_config("xa")
print(cfg.name_en, "|", ", ".join(cfg.tradition), "|", cfg.calendar, "|", cfg.languages)
for code, dc in cfg.document_classes.items():
    levels = " > ".join(h.level for h in dc.hierarchy)
    print(f"  {code:<12} {dc.label:<12} basic unit: {dc.basic_unit:<9} {levels}")
print("enacting formula:", cfg.enacting_formulae[0].text if cfg.enacting_formulae else None)

Commonwealth of Atlantis | common_law | gregorian | ['eng']
  act          Act of the Assembly basic unit: section   higher > higher > basic > subdivision > subdivision > subdivision > grouping
enacting formula: BE IT ENACTED by the Assembly of Atlantis, and by the authority of the same, as follows:


## 2. Bluebell to Akoma Ntoso

[Bluebell](https://github.com/laws-africa/bluebell) is a plain-text grammar for Akoma
Ntoso: keyword lines (`PART`, `SECTION`, `SUBSECTION`) and indented text. The model
writes Bluebell rather than XML, because a Bluebell parse is structurally valid by
construction. The synthetic corpora are kept in this form.

In [3]:
from codify.akn.bluebell import bluebell_to_akn

source = (FIXTURES / "xa" / "freedom-of-information-2001.bluebell").read_text()
print(source[:700])

PREFACE

  No. 3 of 2001

  Freedom of Information Act, 2001

  An Act to confer on the public a general right of access to information held by public authorities, to make provision for the refusal of access on limited grounds, and for connected purposes.

PREAMBLE

  BE IT ENACTED by the Assembly of Atlantis, and by the authority of the same, as follows:

BODY

SECTION 1 - Short title

  This Act may be cited as the Freedom of Information Act, 2001.

SECTION 2 - Right to information

  SUBSECTION (1)

    Every person has the right, on making a request to a public authority, to be informed in writing whether the authority holds the information requested and, if it does, to have that informa


In [4]:
akn_xml, source_map, errors = bluebell_to_akn(
    source, country="xa", doctype="act", date="2001", number="3"
)
assert not errors, errors
print(f"{len(akn_xml):,} characters of XML; {len(source_map)} source anchors")
print(akn_xml[:1200])

4,336 characters of XML; 9 source anchors
<akomaNtoso xmlns="http://docs.oasis-open.org/legaldocml/ns/akn/3.0">
  <act name="act">
    <meta>
      <identification source="#cobalt">
        <FRBRWork>
          <FRBRthis value="/akn/xa/act/2001/3"/>
          <FRBRuri value="/akn/xa/act/2001/3"/>
          <FRBRalias value="Untitled" name="title"/>
          <FRBRdate date="2001" name="Generation"/>
          <FRBRauthor href=""/>
          <FRBRcountry value="xa"/>
          <FRBRnumber value="3"/>
        </FRBRWork>
        <FRBRExpression>
          <FRBRthis value="/akn/xa/act/2001/3/eng"/>
          <FRBRuri value="/akn/xa/act/2001/3/eng"/>
          <FRBRdate date="2026-09-18" name="Generation"/>
          <FRBRauthor href=""/>
          <FRBRlanguage language="eng"/>
        </FRBRExpression>
        <FRBRManifestation>
          <FRBRthis value="/akn/xa/act/2001/3/eng"/>
          <FRBRuri value="/akn/xa/act/2001/3/eng"/>
          <FRBRdate date="2026-09-18" name="Generation"

Each source anchor ties a Bluebell line to the eId the parser gave it, which is how an
editor maps a click in the text to an element in the XML.

In [5]:
for a in source_map[:8]:
    print(f"  line {a.line:>3}  {a.eid}")

  line  15  sec_1
  line  19  sec_2
  line  21  sec_2__subsec_1
  line  25  sec_2__subsec_2
  line  29  sec_3
  line  33  sec_4
  line  35  sec_4__subsec_1
  line  39  sec_4__subsec_2


## 3. The typed document

`parse_akn` reads any AKN 3.0 act into a `Document`: FRBR identifiers, the body as a
tree of elements (part, chapter, section, subsection, paragraph, point), each carrying
its eId, number, heading and text.

In [6]:
from codify.akn import parse_akn

doc = parse_akn(akn_xml)
print("work      ", doc.frbr_work_uri)
print("expression", doc.frbr_expression_uri)
print("language  ", doc.language, "| expression date", doc.expression_date)


def walk(elements, depth=0):
    for el in elements:
        label = f"{el.akn_type} {el.number or ''}".strip()
        text = (el.text or el.intro)[:60].replace("\n", " ")
        print(f"{'  ' * depth}{label:<16} {el.akn_eid:<28} {el.heading or text}")
        walk(el.children, depth + 1)


walk(doc.body)

work       /akn/xa/act/2001/3
expression /akn/xa/act/2001/3/eng
language   eng | expression date 2026-09-18
section 1        sec_1                        Short title
section 2        sec_2                        Right to information
  subsection (1)   sec_2__subsec_1              Every person has the right, on making a request to a public 
  subsection (2)   sec_2__subsec_2              A public authority shall comply with a request under subsect
section 3        sec_3                        Publication of legislation
section 4        sec_4                        Grounds for refusal
  subsection (1)   sec_4__subsec_1              A public authority may refuse a request only where the infor
  subsection (2)   sec_4__subsec_2              Information is exempt if its disclosure would, or would be l
section 5        sec_5                        Appeals


An eId is the provision's address inside the work: `sec_3__subsec_2` is subsection (2)
of section 3, wherever the act is later rendered, searched or cited. `parse_eid` reads
one back into its parts.

In [7]:
from codify.akn import parent_eid, parse_eid

eid = "sec_3__subsec_2"
print(parse_eid(eid))
print("parent:", parent_eid(eid))

[EidContainer(prefix='sec', number='3', raw='sec_3', kind='section'), EidContainer(prefix='subsec', number='2', raw='subsec_2', kind=None)]
parent: sec_3


## 4. Validation and round trip

`validate_akn` checks the XML against the OASIS schema. `to_akn` writes a `Document`
back out; `akn_xml_to_bluebell` goes the other way, which is what an editor shows.

In [8]:
from codify.akn import to_akn, validate_akn
from codify.akn.bluebell import akn_xml_to_bluebell

validate_akn(akn_xml)  # raises DocumentInvalid on failure
print("schema-valid")
assert parse_akn(to_akn(doc)).frbr_expression_uri == doc.frbr_expression_uri
print(akn_xml_to_bluebell(akn_xml)[:500])

schema-valid
PREFACE

  No. 3 of 2001

  Freedom of Information Act, 2001

  An Act to confer on the public a general right of access to information held by public authorities, to make provision for the refusal of access on limited grounds, and for connected purposes.

PREAMBLE

  BE IT ENACTED by the Assembly of Atlantis, and by the authority of the same, as follows:

BODY

SECTION 1 - Short title

  This Act may be cited as the Freedom of Information Act, 2001.

SECTION 2 - Right to information

  SUBSECTI


## 5. From plain text, with the model

A real source is not Bluebell: it is a PDF or the text pulled out of one. Structuring
is anchor-driven. A deterministic scan finds the structural markers the jurisdiction
config declares (here `PART` and `Section` lines), builds a skeleton, and the model
fills in the text of each anchored unit. The scan is the half you inspect first when
an ingest produced too little.

Text lane rather than PDF: the shipped `xa` PDFs carry no text layer, and the vision
route reads their margin section numbers as parts with no sections (a known gap), so a
short plain-text act is used instead. The scan below needs no model.

In [9]:
from codify.quality.corpus_scan import scan_text

ACT = """\
No. 9 of 2015

Coastal Lights Act, 2015

An Act to provide for the maintenance of coastal lights and for connected purposes.

[Assented to 3rd March, 2015]

PART I
PRELIMINARY

Section 1
This Act may be cited as the Coastal Lights Act.

Section 2
In this Act, unless the context otherwise requires, "light" means a lighthouse,
beacon or buoy maintained under this Act, and "Keeper" means the Keeper of Lights
appointed under section 3.

PART II
THE KEEPER OF LIGHTS

Section 3
The Minister shall appoint a Keeper of Lights, who shall hold office for a term of
five years and may be reappointed.

Section 4
The Keeper shall maintain a register of every light, and shall publish the register
in the Gazette not later than the thirty-first day of March in each year.

Section 5
A person who extinguishes, obscures or removes a light without the written consent
of the Keeper commits an offence and is liable on conviction to a fine.
"""

scan = scan_text(ACT, config=cfg, country="xa", doctype="act")
print(f"anchors {scan.anchors}, basic units {scan.basic_units}, by kind {scan.by_kind}")
print("by pass", scan.by_pass)

anchors 7, basic units 5, by kind {'part': 2, 'section': 5}
by pass {'regex': 7}


Now the full pass. `ingest_text` streams events; `Complete` carries the AKN. A short act
like this is one model call and costs well under a cent. The warning it prints is the
model returning text for the part headings, which own no body; the fill drops it.

In [10]:
import os

from codify.core.llm import create_llm_client
from codify.pipeline.events import Complete, Failed, ValidationIssued
from codify.pipeline.formats.pdf import ingest_text

llm = create_llm_client(
    base_url=os.environ["LITELLM_BASE_URL"],
    api_key=os.environ["LITELLM_API_KEY"],
    model=os.environ.get("LITELLM_MODEL", "gemini-3.7-flash"),
    telemetry_mode="direct",
)

traces, findings, result = [], [], None
async for event in ingest_text(ACT, "xa", llm=llm, name="coastal-lights", on_scan=traces.append):
    print(" ", type(event).__name__)
    if isinstance(event, ValidationIssued):
        findings.append(event.issue)
    elif isinstance(event, Complete):
        result = event.akn_xml
    elif isinstance(event, Failed):
        raise RuntimeError(f"{event.stage}: {event.error}")

  MetadataExtracted
  AnchorsDetected


[warning  ] body_fill_unowned_targets      chunk=w0 eids=['part_II']


  StructureProgress
  Structured
  Parsed


  Enriched
  Enriched
  Enriched
  Enriched
  Enriched
  Enriched
  Enriched
  Enriched
  Enriched
  Enriched
  Enriched
  Enriched
  Complete


The trace is the scan's account of itself: the anchors it found, the coverage it
measured (numbers seen in the text against numbers anchored), and the scaffold the model
was asked to fill.

In [11]:
trace = traces[0]
print("coverage:", trace.coverage)
print(trace.scaffold)

coverage: AnchorCoverage(kind='section', ratio=1.0, captured=frozenset({'2', '3', '5', '1', '4'}), expected=frozenset({'2', '3', '5', '1', '4'}), masked=0, unclosed=0)
PREFACE
  No. 9 of 2015
  Coastal Lights Act, 2015
  LONGTITLE An Act to provide for the maintenance of coastal lights and for connected purposes.
  [Assented to 3rd March, 2015]

BODY
  PART I - PRELIMINARY

    SECTION 1

    SECTION 2

  PART II - THE KEEPER OF LIGHTS

    SECTION 3

    SECTION 4

    SECTION 5




In [12]:
validate_akn(result)
for f in findings:
    print("finding:", f)
structured = parse_akn(result)
print(structured.frbr_work_uri)
walk(structured.body)

/akn/xa/act/2015/9
part I           part_I                       PRELIMINARY
  section 1        part_I__sec_1                This Act may be cited as the Coastal Lights Act.
  section 2        part_I__sec_2                In this Act, unless the context otherwise requires, "light" 
part II          part_II                      THE KEEPER OF LIGHTS
  section 3        part_II__sec_3               The Minister shall appoint a Keeper of Lights, who shall hol
  section 4        part_II__sec_4               The Keeper shall maintain a register of every light, and sha
  section 5        part_II__sec_5               A person who extinguishes, obscures or removes a light witho


The same run from a shell writes every one of these artefacts into a bundle directory:

```
uv run codify ingest-one act.txt --jurisdiction xa --out bundle/
```

Next: [Codify 201](codify-201.ipynb) stores documents in Postgres and searches them.